# Baseline: per-generator XGBoost

What is tested: the current best pipeline, reproduced in this folder as the
clean baseline for future experiments (hyperparameters, seed averaging,
recency weighting).

Protocol: one `xgb.XGBClassifier` per generator (14 models) on the 193
features (186 shared + 7 `local_*`), train on cases starting 2015-2021,
early stopping on the last 90 days of the *train* period (never the
validation set), score on cases starting 2022. Hyperparameters:
`n_estimators=400, learning_rate=0.05, max_depth=7, subsample=0.8,
colsample_bytree=0.8, tree_method='hist', eval_metric='auc',
early_stopping_rounds=25, random_state=0`.

Feature engineering is copied verbatim from
`notebooks/eda_unit_commitment.ipynb` / `per_generator_xgboost.ipynb`.

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import roc_auc_score

HOURS = 168

In [2]:
DATA = "../data/kernel"
EXT = "../data/extended"

price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))


def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")
target = target.sort_values("starttime").reset_index(drop=True)

GENERATORS = sorted(
    {c.removeprefix("result_committed_").rsplit("_t", 1)[0]
     for c in target.columns if c.startswith("result_committed")}
)

connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if e["to"] == name and e["to_type"] == kind and e["from_type"] in {"reservoir", "tunnel"}:
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


FEEDERS = {}
for gen in GENERATORS:
    plant = next(e["to"] for e in connections
                 if e["from"] == gen and e["from_type"] == "generator" and e["to_type"] == "plant")
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))

print("cases:", len(target), " generators:", len(GENERATORS))

cases: 2922  generators: 14


## Feature engineering

What is tested: the 193-column feature set — `feature_matrix()` builds the
186 shared columns (calendar, price context, per-reservoir state,
per-inflow-source, constraints), `generator_features()` adds the 7
`local_*` aggregates over each generator's intake reservoirs.

In [3]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (("mean", price.mean(1)), ("std", price.std(1)),
                    ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    for off in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{off:+d}", price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)])

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(inf_wide.columns):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(f"init_margin_to_minvol__{name}", initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j])

    return pd.DataFrame(F, dtype=np.float32)


def generator_features(base, gen):
    X = base.copy()
    feeders = FEEDERS[gen]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_init = sum(base[f"init_vol__{r}"] for r in feeders)
    value = sum(base[f"term_val__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cum = sum(base[f"inflow_cum_Mm3__{r}"] for r in feeders)
    rem = sum(base[f"inflow_rem_Mm3__{r}"] for r in feeders)

    X["local_init_frac"] = total_init / capacity
    X["local_term_val"] = value
    X["local_price_minus_val"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cum_frac"] = cum / capacity
    X["local_rem_frac"] = rem / capacity
    X["local_available_frac"] = (total_init + cum + rem) / capacity
    return X


def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)

## Fit and score

What is tested: train on cases starting before 2022-01-01, early stopping
on the last 90 days of that train period, score on the 2022 cases.
Per-generator AUC and the pooled micro AUC (the competition metric) are
printed.

In [4]:
split = pd.Timestamp("2022-01-01", tz="UTC")
train_cases = target[target["starttime"] < split].reset_index(drop=True)
val_cases = target[target["starttime"] >= split].reset_index(drop=True)
stop_cases = train_cases[
    train_cases["starttime"] >= split - pd.Timedelta(days=90)
].reset_index(drop=True)
print("train cases:", len(train_cases), " stop cases:", len(stop_cases),
      " val cases:", len(val_cases))

Xtr_base = feature_matrix(train_cases)
Xst_base = feature_matrix(stop_cases)
Xva_base = feature_matrix(val_cases)
print("feature matrix:", Xtr_base.shape)

train cases: 2557  stop cases: 90  val cases: 365
feature matrix: (429576, 186)


In [5]:
preds, auc_rows = {}, []
for gen in GENERATORS:
    Xtr = generator_features(Xtr_base, gen)
    Xst = generator_features(Xst_base, gen)
    Xva = generator_features(Xva_base, gen)
    ytr = labels_for(train_cases, gen)
    yst = labels_for(stop_cases, gen)
    yva = labels_for(val_cases, gen)

    if np.unique(ytr).size == 1:
        preds[gen] = np.full(len(yva), float(ytr[0]), dtype=np.float32)
        auc_rows.append({"generator": gen, "trees": 0, "AUC": None})
        continue

    clf = xgb.XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, tree_method="hist",
        eval_metric="auc", early_stopping_rounds=25, random_state=0,
    )
    clf.fit(Xtr, ytr, eval_set=[(Xst, yst)], verbose=False)
    preds[gen] = clf.predict_proba(Xva)[:, 1].astype(np.float32)
    auc_rows.append({"generator": gen, "trees": clf.best_iteration + 1,
                     "AUC": roc_auc_score(yva, preds[gen])})
    print(f"{gen:16s} AUC {auc_rows[-1]['AUC']:.4f}  ({auc_rows[-1]['trees']} trees)")

y_all = np.concatenate([labels_for(val_cases, g) for g in GENERATORS])
p_all = np.concatenate([preds[g] for g in GENERATORS])
print("micro AUC:", round(roc_auc_score(y_all, p_all), 4))
pd.DataFrame(auc_rows).set_index("generator").round(4)

Byrte_G1         AUC 0.9951  (400 trees)
Haukeli_G1       AUC 0.9352  (398 trees)
Hogga_G1         AUC 0.9722  (400 trees)
Kjela_G1         AUC 0.9789  (400 trees)
Lio_G1           AUC 0.9225  (400 trees)
Songa_G1         AUC 0.9966  (133 trees)
Tokke_G1         AUC 0.9928  (400 trees)
Tokke_G2         AUC 0.9839  (400 trees)
Tokke_G3         AUC 0.9886  (400 trees)
Tokke_G4         AUC 0.9810  (399 trees)
Vesle Kjela_G1   AUC 0.9815  (400 trees)
Vinje_G1         AUC 0.9835  (400 trees)
Vinje_G2         AUC 0.9765  (398 trees)
Vinje_G3         AUC 0.9701  (400 trees)
micro AUC: 0.9802


,trees,AUC
generator,,
Byrte_G1,400,0.9951
Haukeli_G1,398,0.9352
Hogga_G1,400,0.9722
Kjela_G1,400,0.9789
Lio_G1,400,0.9225
Songa_G1,133,0.9966
Tokke_G1,400,0.9928
Tokke_G2,400,0.9839
Tokke_G3,400,0.9886
